In [1]:
import numpy as np
import pandas as pd
from collections import Counter

from rdflib import Graph, Literal, RDF, RDFS, URIRef, Namespace, BNode
from rdflib.namespace import SDO, XSD, PROV
import re
from tqdm import tqdm


In [2]:

g = Graph()
products_ns = Namespace('http://products-kg.org/')
products_scheme = Namespace('http://products-kg.org/scheme/')

g.bind("schema", SDO)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
g.bind("prov", PROV)
g.bind("pkg", products_ns)
g.bind("pkgs", products_scheme)

# Explicitly define classes that are used in the graph
g.add((SDO.Product, RDF.type, RDFS.Class))
g.add((SDO.Offer, RDF.type, RDFS.Class))
g.add((SDO.PropertyValue, RDF.type, RDFS.Class))
g.add((SDO.AggregateRating, RDF.type, RDFS.Class))
g.add((SDO.QuantitativeValue, RDF.type, RDFS.Class))
g.add((PROV.Entity, RDF.type, RDFS.Class))
g.add((SDO["OnlineStore"], RDF.type, RDFS.Class))
g.add((SDO["OnlineStore"], RDFS.comment, Literal("An eCommerce site", lang='en')))

# Basic properties(note that these are not all properties used in the graph, these are only required properties. More other properties could be used during graph enrichment)
g.add((SDO.additionalProperty, RDF.type, RDF.Property))
g.add((SDO.additionalProperty, RDFS.domain, SDO.Product))
g.add((SDO.additionalProperty, RDFS.range, SDO.PropertyValue))

g.add((SDO.name, RDF.type, RDF.Property))
g.add((SDO.name, RDFS.domain, SDO.Product))
g.add((SDO.name, RDFS.range, XSD.string))

g.add((SDO.description, RDF.type, RDF.Property))
g.add((SDO.description, RDFS.domain, SDO.Product))
g.add((SDO.description, RDFS.range, XSD.string))

g.add((SDO.offers, RDF.type, RDF.Property))
g.add((SDO.offers, RDFS.domain, SDO.Product))
g.add((SDO.offers, RDFS.range, SDO.Offer))

g.add((SDO.price, RDF.type, RDF.Property))
g.add((SDO.price, RDFS.domain, SDO.Offer))
g.add((SDO.price, RDFS.range, XSD.float))

g.add((PROV.wasDerivedFrom, RDF.type, RDF.Property))
g.add((PROV.wasDerivedFrom, RDFS.domain, SDO.Product))
g.add((PROV.wasDerivedFrom, RDFS.range, PROV.Entity))

g.add((products_scheme["fromOnlineStore"], RDF.type, RDF.Property))
g.add((products_scheme["fromOnlineStore"], RDFS.domain, SDO.Product))
g.add((products_scheme["fromOnlineStore"], RDFS.range, SDO["OnlineStore"]))
g.add((products_scheme["fromOnlineStore"], RDFS.comment, Literal("An online store in which product can be found", lang='en')))


print(f"Ontology defined. Graph currently has {len(g)} triples.")
print(g.serialize(destination="../data/ontology_graph.ttl", format="ttl"))


Ontology defined. Graph currently has 30 triples.
[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


C:\Users\denve\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3701: UserWarning: Code: OnlineStore is not defined in namespace SDO
  exec(code_obj, self.user_global_ns, self.user_ns)


# Enriching with Amazon data

In [3]:
df_amazon = pd.read_pickle("../data/amazon_products_cleaned.pkl")
df_amazon.reset_index(drop=True, inplace=True)
df_amazon.head()

,parent_asin,date_first_available,title,description,main_category,average_rating,rating_number,price,features,details
0,B0BT4CWWC9,2023-01-26,Sunshine On My Shoulders: The Best Of John Den...,"Sunshine On My Shoulders is a 2CD, 36-track co...",Digital Music,4.7,502.0,19.98,NaN,{'Package Dimensions': '5.55 x 4.92 x 0.51 inc...
1,B0BS4L5LP6,2023-01-11,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. H...,Digital Music,5.0,1.0,14.97,NaN,"{'Item Weight': '4 Ounces', 'Run time': '1 hou..."
2,B0BSPBBP89,2023-01-20,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music,4.8,34.0,12.99,NaN,{'Package Dimensions': '5.59 x 4.8 x 0.47 inch...
3,B0BPT1369H,2023-02-14,HIGH DRAMA. CD SIGNED-ADAM LAMBERT,Renowned for re-imagining songs from American ...,Digital Music,4.7,207.0,23.35,NaN,"{'Language': 'English', 'Product Dimensions': ..."
4,B0BVYD326Y,2023-02-15,Hamilton Original Broadway Cast Recording (Exp...,Explicit version. 2015 two CD set. The origina...,Digital Music,4.6,56.0,21.86,NaN,{'Package Dimensions': '5.55 x 4.88 x 0.94 inc...


We are loading Schema.org graph to extract properties that we will dynamically use during graph enrichment

In [4]:
sdo_data = Graph()
sdo_data.parse("https://schema.org/version/latest/schemaorg-all-http.ttl", format="turtle")


# Select all properties that have domain of SDO:Product
results = sdo_data.query("""
PREFIX sdo: <http://schema.org/>

SELECT ?property ?name ?description
WHERE
{

    ?property sdo:domainIncludes sdo:Product.
    ?property rdfs:label ?name.
    ?property rdfs:comment ?description.
}
""")

# We save make name+description for embedding of properties, so that we can search for needed properties as we process the data
sdo_property_index = []
sdo_property_text = []
for result in results:
    sdo_property_index.append({
        "url": result.property,
        "name": result.name,
        "comment": result.description
    })
    # convert property name from camelCase to usual word
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', result.name)
    sdo_property_text.append(f"{name}")

    # add name and description as separate vectors with the same metadata. So if search doesn't find the match by label it can still
    # find it by description
    sdo_property_index.append({
        "url": result.property,
        "name": result.name,
        "comment": result.description
    })
    sdo_property_text.append(f"{result.description}")




We are creating a vector store from texts extracted `schema.org` graph

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from torch import sigmoid

C:\Users\denve\miniconda3\envs\KG_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# embedding_model = HuggingFaceEmbeddings(model_name='BAAI/bge-large-en-v1.5')
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

# sdo_embeddings = model.encode(sdo_labels)

vector_store = FAISS.from_texts(sdo_property_text, embedding=embedding_model, metadatas=sdo_property_index)

C:\Users\denve\AppData\Local\Temp\ipykernel_620\746124631.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11884.59it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def find_semantic_match(query, vector_store):
    result = vector_store.similarity_search_with_score(query, k=1)[0]

    if result[1] < 0.65:  # distance threshold
        return result[0].metadata
    return None

# re_ranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', default_activation_function=sigmoid)
#
# def find_semantic_match(query, vector_store) -> dict|None:
#     results = vector_store.similarity_search_with_score(query, k=5)
#
#     pairs = [[query, r[0].page_content] for r in results]
#     scores = re_ranker.predict(pairs)
#
#     # Pair results with their new scores and sort
#     scored_results = sorted(zip(results, scores), key=lambda x: x[1], reverse=True)
#
#
#     best_match, best_score = scored_results[0]
#
#     if best_score > 0.5:
#         return best_match[0].metadata
#     return None



We keep track of all the keys in `details` to later filter our the rare ones

In [8]:
details_key_counter = Counter()
for details in df_amazon['details']:
    if pd.isna(details):
        continue

    for key in details.keys():
        details_key_counter.update([key.lower().strip()])

print(details_key_counter)

Counter({'date first available': 81214, 'manufacturer': 62243, 'item model number': 39703, 'brand': 37174, 'item weight': 36247, 'best sellers rank': 36117, 'department': 33212, 'color': 28875, 'product dimensions': 28321, 'country of origin': 23800, 'material': 22235, 'package dimensions': 20615, 'special feature': 13594, 'style': 12001, 'included components': 8969, 'size': 8944, 'number of items': 8827, 'shape': 8588, 'age range (description)': 6473, 'special features': 6463, 'theme': 6212, 'manufacturer part number': 6044, 'pattern': 5709, 'number of pieces': 5605, 'item dimensions lxwxh': 5509, 'product care instructions': 5452, 'unit count': 5382, 'batteries required?': 4848, 'part number': 4785, 'model name': 4366, '': 3941, 'mounting type': 3897, 'room type': 3810, 'recommended uses for product': 3258, 'batteries included?': 3190, 'finish type': 3162, 'form factor': 3159, 'occasion': 3074, 'brand name': 2885, 'other display features': 2754, 'compatible devices': 2655, 'model': 2

Define additional entities related to the Amazon dataset and parse all the items

In [9]:
g.add((products_ns["source/1"], RDF.type, PROV.Entity))
g.add((products_ns["source/1"], RDFS.label, Literal("milistu/AMAZON-Products-2023")))
g.add((products_ns["source/1"], SDO.url, URIRef("https://huggingface.co/datasets/milistu/AMAZON-Products-2023")))

g.add((products_ns["online_store/1"], RDF.type, SDO["OnlineStore"]))
g.add((products_ns["online_store/1"], RDFS.label, Literal("Amazon", datatype=XSD.string)))
g.add((products_ns["online_store/1"], SDO.url, URIRef("https://www.amazon.com/")))

for index, row in tqdm(df_amazon[0:50].iterrows()):
    product_uri = products_ns[f'product/{index}']
    offer_uri = products_ns[f'offer/{index}']

    g.add((product_uri, RDF.type, SDO.Product))
    g.add((offer_uri, RDF.type, SDO.Offer))
    g.add((product_uri, SDO.offers, offer_uri))

    g.add((product_uri, products_scheme["hasOnlineStore"], products_ns["online_store/1"]))
    g.add((product_uri, PROV.wasDerivedFrom, products_ns["source/1"]))

    if pd.notna(row['parent_asin']):
        g.add((product_uri, SDO['asin'], Literal(row['parent_asin'], datatype=XSD.string)))

    if pd.notna(row['date_first_available']):
        g.add((offer_uri, SDO.availabilityStarts, Literal(row['date_first_available'], datatype=XSD.date)))

    if pd.notna(row['title']):
        g.add((product_uri, RDFS.label, Literal(row['title'], datatype=XSD.string)))

    if pd.notna(row['description']):
        g.add((product_uri, SDO.description, Literal(row['description'], datatype=XSD.string)))

    if pd.notna(row['main_category']):
        g.add((product_uri, SDO.category, Literal(row['main_category'], datatype=XSD.string)))

    if pd.notna(row['average_rating']):
        g.add((product_uri, SDO.aggregateRating, Literal(row['average_rating'], datatype=XSD.float)))

    if pd.notna(row['price']):
        g.add((offer_uri, SDO.price, Literal(row['price'], datatype=XSD.float)))
        g.add((offer_uri, SDO.priceCurrency, Literal("USD", datatype=XSD.string)))

    if pd.notna(row['features']):
        feature_node = BNode()
        g.add((product_uri, SDO.additionalProperty, feature_node))
        g.add((feature_node, RDF.type, SDO.PropertyValue))
        g.add((feature_node, SDO.name, Literal("Features", datatype=XSD.string)))
        g.add((feature_node, SDO.value, Literal(row['features'], datatype=XSD.string)))


    if not isinstance(row['details'], dict):
        continue

    for (key, value) in row['details'].items():
        added_details = []
        key = key.lower().strip()

        if not (isinstance(key, str) and isinstance(value, str)):
            continue

        if len(key) == 0 or len(value) == 0:
            continue

        # we skip package dimensions because there are no relevant Properties for that, and it is not directly about the product
        if "package dimensions" in key:
            continue

        # we already have date first available, so skip it
        if "date first available" in key:
            continue

        if "dimensions" in key:
            try:
                size, weight = None, None
                split = value.strip().split(";")
                if len(split) == 2:
                    size, weight = split
                else:
                    size = split[0]

                if not weight is None:
                    weight_val, weight_unit = weight.strip().split()
                    weight_node = BNode()
                    g.add((product_uri, SDO.weight, weight_node))
                    g.add((weight_node, RDF.type, SDO.QuantitativeValue))
                    g.add((weight_node, SDO.value, Literal(weight_val, datatype=XSD.float)))
                    g.add((weight_node, SDO.unitText, Literal(weight_unit, datatype=XSD.string)))

                g.add((product_uri, SDO.size, Literal(size, datatype=XSD.string)))
                continue
            except ValueError:
                continue


        matched_prop = find_semantic_match(key, vector_store  )
        if not matched_prop is None:
            # we always consider value type to be string for simplicity,
            # but also it doesn't make sense to create classes like `Country` in the products graph.
            # We have to use property by name, because of prefix mismatch. SDO has namespace with https, but query returns http
            g.add((product_uri, SDO[matched_prop.get('name')], Literal(value, datatype=XSD.string)))
        else:
            # if the detail specification is rare, it is most likely a corrupted string, so we don't add it
            if details_key_counter.get(key, 0) < 5:
                continue
            # if there are no similar properties in schema.org we use sdo:additionalProperty
            spec_node = BNode()
            g.add((product_uri, SDO.additionalProperty, spec_node))
            g.add((spec_node, RDF.type, SDO.PropertyValue))
            g.add((spec_node, SDO.name, Literal(key, datatype=XSD.string)))
            g.add((spec_node, SDO.value, Literal(value, datatype=XSD.string)))


0it [00:00, ?it/s]C:\Users\denve\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3701: UserWarning: Code: asin is not defined in namespace SDO
  exec(code_obj, self.user_global_ns, self.user_ns)
50it [00:03, 13.25it/s]


# Enriching the graph with WDC dataset

In [10]:
df_wdc = pd.read_pickle("../data/wdc_products_cleaned.pkl")
# reset the index to align with amazon dataset
df_wdc.set_index(np.arange(len(df_amazon), len(df_amazon) + len(df_wdc), 1, dtype=int), inplace=True)
df_wdc.head()

,category,identifiers,title,description,brand,price,keyValuePairs,url,store name,store url
81359,Tools and Home Improvement,{'/sku': 'gad00192'},"""Bee line Dragway Sign"" ""Drag Racing - Signs f...","""Bee Line Dragway in Arizona - Smell the smoke...",NaN,"""USD"", ""99.95""",NaN,http://garageart.com/products_cat2.asp?Categor...,garageart,garageart.com
81360,Sports and Outdoors,{'/sku': 'underarmourftw1264304001'},"""Under Armour SpeedForm Gemini Men's Team Trai...","""Slide the UA SpeedForm Gemini on & you immedi...","""Baseball Monkey""@en","""USD""@en, ""129.99""@en",NaN,https://www.baseballmonkey.com/homerun-footwea...,baseballmonkey,www.baseballmonkey.com
81361,Other Electronics,"{'/gtin8': '43233205', '/mpn': 'cese1gscu'}","""Sophos Central Endpoint Standard - competitiv...","""Sophos Cloud is the only integrated security ...",NaN,"""$"", ""22.70""",NaN,https://www.cdw.com/shop/products/Sophos-Cloud...,cdw,www.cdw.com
81362,Computers and Accessories,{'/mpn': 'hardkernelodroidxu4'},"""Odroid XU4 - 8 Core Odroid computer (inc PSU)...",networking especially with faster booting USB ...,"""Hard Kernel""@en","""91.19€""@en, ""EUR""@en",NaN,https://lilliputdirect.com/index.php?_route_=o...,lilliputdirect,lilliputdirect.com
81363,Sports and Outdoors,{'/sku': 'rwlbhr16h2j'},"""Rawlings Velo Two-Tone Home Junior Batting He...","""With its eye-catching finish and ultra-cushio...","""Baseball Monkey""@en","""USD""@en, ""42.99""@en",NaN,https://www.baseballmonkey.com/equipment/homer...,baseballmonkey,www.baseballmonkey.com


In [11]:
keyValuePairs_key_counter = Counter()
for details in df_wdc['keyValuePairs']:
    if pd.isna(details):
        continue

    for key in details.keys():
        keyValuePairs_key_counter.update([key.lower().strip()])

print(keyValuePairs_key_counter)

Counter({'category': 1342, 'part number': 1214, 'sub-category': 1210, 'generation': 1210, 'products id': 986, '': 863, 'capacity': 620, 'manufacturer': 539, 'spindle speed': 458, 'form factor': 403, 'interface type': 395, 'data transfer rate': 365, 'type': 343, 'interface': 328, 'pre-failure warranty': 242, 'enclosure': 227, 'product id': 224, 'hot swap tray': 211, 'product type': 210, 'seek time': 201, 'drive dimensions': 192, 'pax-code': 188, 'item code': 186, 'version': 184, 'processor qty': 180, 'ports': 179, 'clock speed': 176, 'processor type': 175, 'processor core': 175, 'official release date': 175, 'genre': 173, 'hotswap': 172, 'processor socket': 172, 'l2 cache': 165, 'thermal design power': 163, '64-bit processing': 161, 'process technology': 146, 'catalog no.': 137, 'rotational speed': 137, 'ad id': 128, 'posted': 128, 'expiry': 128, 'status': 128, 'external data transfer': 123, 'details': 106, 'weight': 100, 'l3 cache': 95, 'enclosure type': 92, 'bus speed': 92, 'tape type

In [12]:
identifiers_counter = Counter()
for details in df_wdc['identifiers']:
    if pd.isna(details):
        continue

    for key in details.keys():
        identifiers_counter.update([key])

print(identifiers_counter)

Counter({'/sku': 14446, '/mpn': 4983, '/gtin8': 1774, '/gtin13': 1293, '/productID': 666, '/identifier': 74, '/gtin12': 55, '/gtin14': 31})


In [13]:
# We manually create mapping for identifiers and relevant schema properties, to avoid running semantics search more than necessary
identifiers_mapping = {
    "/sku": SDO.sku,
    "/mpn": SDO.mpn,
    "/gtin8": SDO.gtin8,
    "/gtin12": SDO.gtin12,
    "/gtin13": SDO.gtin13,
    "/gtin14": SDO.gtin14,
    "/productID": SDO.productID,
    "/identifier": SDO.identifier,
}

In [14]:
def extract_price(input_string):
    # Mapping for common symbols to ISO 4217 codes
    symbol_to_iso = {
        "$": "USD",
        "€": "EUR",
        "£": "GBP",
        "¥": "JPY",
        "₹": "INR"
    }

    # A matching pattern for price, considering commas as delimiters and dot as fraction separator
    price_pattern = r'(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?'

    # Matches 3-letter uppercase codes or common symbols
    currency_pattern = r'[A-Z]{3}|[\$\€\£\¥]'

    # Extract price and currency
    price_match = re.search(price_pattern, input_string)
    currency_match = re.search(currency_pattern, input_string)

    if price_match is None:
        return None, None

    # Clean price value
    price_val = price_match.group(0) if price_match else None
    if price_val:
        price_val = price_val.replace(',', '')

    if currency_match is None:
        return price_val, currency_match

    # Map to ISO standard code
    raw_currency = currency_match.group(0) if currency_match else None
    iso_currency = symbol_to_iso.get(raw_currency, raw_currency)

    return float(price_val), iso_currency



In [15]:
g.add((products_ns["source/2"], RDF.type, PROV.Entity))
g.add((products_ns["source/2"], RDFS.label, Literal("Web Data Commons, Product Data Corpus (English)")))
g.add((products_ns["source/2"], SDO.url, URIRef("http://webdatacommons.org/largescaleproductcorpus/v2/index.html")))

stores_unique = set()
store_id = 1 # we start from 2, because 1 is already Amazon
for index, row in tqdm(df_wdc[0:50].iterrows()):
    store_id += 1
    product_uri = products_ns[f'product/{index}']
    offer_uri = products_ns[f'offer/{index}']
    store_uri = products_ns[f"online_store/{store_id}"]

    g.add((product_uri, RDF.type, SDO.Product))
    g.add((offer_uri, RDF.type, SDO.Offer))
    g.add((product_uri, SDO.offers, offer_uri))

    g.add((product_uri, PROV.wasDerivedFrom, products_ns["source/1"]))

    # add store if it doesn't yet exist in the graph
    if row["store name"] not in stores_unique:
        g.add((store_uri, RDF.type, SDO["OnlineStore"]))
        g.add((store_uri, RDFS.label, Literal(row["store name"], datatype=XSD.string)))
        g.add((store_uri, SDO.url, URIRef(row["store url"])))
        stores_unique.add(row["store name"])

    g.add((product_uri, products_scheme["fromOnlineStore"], store_uri))
    g.add((product_uri, SDO.url, URIRef(row['url'])))

    for identifier, id_value in row['identifiers'].items():
        g.add((product_uri, identifiers_mapping[identifier], Literal(id_value, datatype=XSD.string)))

    if pd.notna(row['title']):
        g.add((product_uri, RDFS.label, Literal(row['title'], datatype=XSD.string)))

    if pd.notna(row['description']):
        g.add((product_uri, SDO.description, Literal(row['description'], datatype=XSD.string)))

    if pd.notna(row['category']):
        g.add((product_uri, SDO.category, Literal(row['category'], datatype=XSD.string)))

    if pd.notna(row['brand']):
        g.add((product_uri, SDO.brand, Literal(row['brand'], datatype=XSD.string)))


    if pd.notna(row['price']):
        price, currency = extract_price(row["price"])

        if price is None:
            continue
        g.add((offer_uri, SDO.price, Literal(price, datatype=XSD.float)))

        if currency is None:
            continue
        g.add((offer_uri, SDO.priceCurrency, Literal(currency, datatype=XSD.string)))


    if not isinstance(row['keyValuePairs'], dict):
        continue

    for (key, value) in row['keyValuePairs'].items():
        added_details = []
        key = key.lower().strip()

        if not (isinstance(key, str) and isinstance(value, str)):
            continue

        if len(key) == 0 or len(value) == 0:
            continue

        if "dimensions" in key:
            try:
                size, weight = None, None
                split = value.strip().split(";")
                if len(split) == 2:
                    size, weight = split
                else:
                    size = split[0]

                if weight is not None:
                    weight_val, weight_unit = weight.strip().split()
                    weight_node = BNode()
                    g.add((product_uri, SDO.weight, weight_node))
                    g.add((weight_node, RDF.type, SDO.QuantitativeValue))
                    g.add((weight_node, SDO.value, Literal(weight_val, datatype=XSD.float)))
                    g.add((weight_node, SDO.unitText, Literal(weight_unit, datatype=XSD.string)))

                g.add((product_uri, SDO.size, Literal(size, datatype=XSD.string)))
                continue
            except ValueError:
                continue

        matched_prop = find_semantic_match(key, vector_store)
        if not matched_prop is None:
            # we always consider value type to be string for simplicity,
            # but also it doesn't make sense to create classes like `Country` in the products graph.
            g.add((product_uri, SDO[matched_prop.get('name')], Literal(value, datatype=XSD.string)))
        else:
            # if the detail specification is rare, it is most likely a corrupted string, so we don't add it
            if details_key_counter.get(key, 0) < 5:
                continue
            # if there are no similar properties in schema.org we use sdo:additionalProperty
            spec_node = BNode()
            g.add((product_uri, SDO.additionalProperty, spec_node))
            g.add((spec_node, RDF.type, SDO.PropertyValue))
            g.add((spec_node, SDO.name, Literal(key, datatype=XSD.string)))
            g.add((spec_node, SDO.value, Literal(value, datatype=XSD.string)))


50it [00:00, 69.47it/s]


In [16]:
g.serialize(destination="../data/graph.ttl", format='ttl')

<Graph identifier=Na5c8006e880c4b4fac0824e78250c7a2 (<class 'rdflib.graph.Graph'>)>

---
# Validating Graph with SHACL
We can use `pyshacl` library to validated out graph against the shape file

In [17]:
from pyshacl import validate

shape_g = Graph().parse("SHACL.ttl")
conforms, results_graph, results_text = validate(data_graph=g, shacl_graph=shape_g)
print(results_text)

Validation Report
Conforms: False
Results (29):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:path rdfs:url ]
	Focus Node: <http://products-kg.org/online_store/10>
	Result Path: rdfs:url
	Message: Less than 1 values on <http://products-kg.org/online_store/10>->rdfs:url
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:path rdfs:url ]
	Focus Node: <http://products-kg.org/online_store/11>
	Result Path: rdfs:url
	Message: Less than 1 values on <http://products-kg.org/online_store/11>->rdfs:url
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ s